In [ ]:
%pip install gradio
%pip install python-dotenv
%pip install requests
%pip install azure-cognitiveservices-speech
%pip install openai
%pip install ipywidgets

In [ ]:
from dotenv import load_dotenv
import gradio as gr
import os
import azure.cognitiveservices.speech as speechsdk
from openai import AzureOpenAI

load_dotenv()

# Azure OpenAI Setup
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
if endpoint and "/openai/" in endpoint:
    endpoint = endpoint.split("/openai/")[0] + "/"

openai_client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version="2024-02-15-preview"
)
deployment_name = os.getenv("DEPLOY_NAME")

def request_stt(audio_path):
    api_key = os.getenv('SPEECH_KEY')
    region = os.getenv('REGION', "koreacentral")

    if audio_path is None:
        return ''

    speech_config = speechsdk.SpeechConfig(subscription=api_key, region=region)
    speech_config.speech_recognition_language = "ko-KR"

    audio_config = speechsdk.audio.AudioConfig(filename=audio_path)
    speech_recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config, audio_config=audio_config)

    print("Recognizing speech...")
    result = speech_recognizer.recognize_once_async().get()

    if result.reason == speechsdk.ResultReason.RecognizedSpeech:
        print(f"Recognized: {result.text}")
        return result.text
    elif result.reason == speechsdk.ResultReason.NoMatch:
        print("No speech could be recognized")
        return ""
    elif result.reason == speechsdk.ResultReason.Canceled:
        cancellation_details = result.cancellation_details
        print("Speech Recognition canceled: {}".format(cancellation_details.reason))
        if cancellation_details.reason == speechsdk.CancellationReason.Error:
            print("Error details: {}".format(cancellation_details.error_details))
        return ""

    return ""

def chatgpt_response(prompt, history):
    # history 는 messages 형식(list[dict])이며, 비어 있을 수 있음
    if history is None:
        history = []

    prompt = (prompt or "").strip()
    if not prompt:
        # 입력이 없으면 상태를 그대로 유지
        return "", history

    messages = [{"role": "system", "content": "당신은 사용자에게 유용한 정보를 제공하는 친절한 AI 어시스턴트입니다."}]
    messages.extend(history)
    messages.append({"role": "user", "content": prompt})

    try:
        response = openai_client.chat.completions.create(
            model=deployment_name,
            messages=messages,
            max_completion_tokens=500
        )
        bot_message = response.choices[0].message.content
    except Exception as e:
        bot_message = f"오류 발생: {str(e)}"

    history = history + [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": bot_message},
    ]
    # 첫 번째 반환값으로 입력창을 비움
    return "", history

def change_audio(audio_path):
    print("Audio path:", audio_path)
    text = request_stt(audio_path=audio_path)
    return text

with gr.Blocks() as demo:
    gr.Markdown('<h2>Speech & OpenAI Chatbot</h2>')

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown('<h3>STT (음성 인식)</h3>')
            input_mic = gr.Audio(
                label="마이크 입력", sources=["microphone"], type="filepath",
                waveform_options=gr.WaveformOptions(
                    waveform_color="#01C6FF",
                    waveform_progress_color="#01C6FF",
                    skip_length=2
                )
            )
            output_textbox = gr.Textbox(label="인식된 텍스트", placeholder="음성이 변환된 텍스트가 표시됩니다.")
            stt_send_button = gr.Button("인식된 텍스트로 전송", variant="primary")

        with gr.Column(scale=2):
            gr.Markdown('<h3>ChatGPT</h3>')
            chatbot = gr.Chatbot(label="채팅 내역")
            with gr.Row():
                user_input = gr.Textbox(show_label=False, placeholder="메시지를 입력하세요...", scale=8)
                submit_button = gr.Button("전송", scale=1)

            clear_btn = gr.Button("대화 지우기")

    # Events
    # 마이크 입력이 바뀌면 STT 실행 → 인식된 텍스트 표시
    input_mic.change(fn=change_audio, inputs=[input_mic], outputs=[output_textbox])

    # 인식된 텍스트를 OpenAI 로 전송 (전송 후 인식 텍스트박스는 비움)
    stt_send_button.click(fn=chatgpt_response, inputs=[output_textbox, chatbot], outputs=[output_textbox, chatbot])

    # 직접 입력한 텍스트를 OpenAI 로 전송
    submit_button.click(fn=chatgpt_response, inputs=[user_input, chatbot], outputs=[user_input, chatbot])
    user_input.submit(fn=chatgpt_response, inputs=[user_input, chatbot], outputs=[user_input, chatbot])
    clear_btn.click(lambda: [], None, chatbot, queue=False)

demo.launch(share=True)
